# Parámetro de Lindemann en NVT: por pares, estático y dinámico

**Paso 1.** Se calcula la g(r) total y las parciales del frame inicial para elegir el radio de corte en un mínimo de la g(r), sabiendo cuántos vecinos incluye.
**Paso 2.** Se calcula el Lindemann con ese `rcut`. Las tres funciones siguen la estructura de tu código original: mismos pares vecinos (`neighbor_list` con `rcut` en el frame 0), evaluación cada `eval_every` frames y promedio desde el frame 0.

| Método | Definición por par |
|---|---|
| **δ por pares** (tu función, sin cambios) | sqrt(⟨r²⟩ − ⟨r⟩²) / ⟨r⟩, con imagen mínima en cada frame |
| **γ_m estático** (Bedanov et al., 1985) | ⟨\|u_i − u_j\|²⟩ / r0², varianza del vector relativo i→j |
| **γ_L dinámico** (Zahn y Maret, 2000) | \|Δu_i(t) − Δu_j(t)\|² / (2 r0²), con Δu(t) = R(t) − R(0) |

**Uso:** ejecuta las celdas 1–5 una vez, rellena la celda 6 y sigue en orden. En la celda 8 eliges la capa de vecinos (el `rcut`).

In [ ]:
# Celda 1 · librerías
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ase.io.trajectory import Trajectory
from ase.neighborlist import neighbor_list

In [ ]:
# Celda 2 · tu función por pares (sin cambios, salvo que ahora devuelve los datos)

def lindemann_analysis(fname, save_interval, eval_every, rcut=3.2,
                        threshold=0.06, plot_spatial=False):
    """
    Calcula el índice de Lindemann (versión por pares) a lo largo de un .traj.

    Parámetros
    ----------
    fname : str
        Nombre base del archivo (sin extensión .traj)
    save_interval : int
        Pasos de MD entre frames guardados
    eval_every : int
        Cada cuántos frames se evalúa/guarda δ
    rcut : float
        Radio de corte para definir pares vecinos (Å)
    threshold : float
        Valor de δ para el cual se reporta el step de cruce
    plot_spatial : bool
        Si True, además plotea el mapa espacial de δ por átomo (acumulado, corrida completa)
    """
    traj = Trajectory(fname + ".traj")
    n_frames = len(traj)
    print(f"Frames totales: {n_frames}")

    ref_atoms = traj[0]
    #assert list(ref_atoms.get_pbc()) == [True, True, False], "Revisa pbc=[T,T,F]"
    cell, pbc_mask = ref_atoms.get_cell(), np.array(ref_atoms.get_pbc(), float)
    ref_pos = ref_atoms.get_positions()

    i_idx, j_idx = neighbor_list("ij", ref_atoms, cutoff=rcut)
    keep = i_idx < j_idx
    i_idx, j_idx = i_idx[keep], j_idx[keep]
    print(f"Pares vecinos (rcut={rcut} Å): {len(i_idx)}")

    def r_ij(atoms):
        d = atoms.positions[j_idx] - atoms.positions[i_idx]
        frac = np.linalg.solve(cell.T, d.T).T
        frac -= np.round(frac) * pbc_mask
        return np.linalg.norm(frac @ cell, axis=1)

    # ── Welford online por par ───────────────────────────────────
    mean_r, M2_r = np.zeros(len(i_idx)), np.zeros(len(i_idx))
    steps_out, mean_out, min_out, max_out = [], [], [], []

    for count, f in enumerate(range(n_frames), start=1):
        r = r_ij(traj[f])
        d = r - mean_r
        mean_r = mean_r + d / count
        M2_r = M2_r + d * (r - mean_r)

        if count % eval_every == 0 or f == n_frames - 1:
            delta_pair = np.sqrt(np.maximum(M2_r / count, 0)) / mean_r
            steps_out.append(f * save_interval)
            mean_out.append(delta_pair.mean())
            min_out.append(delta_pair.min())
            max_out.append(delta_pair.max())

    steps_out = np.array(steps_out)
    mean_out  = np.array(mean_out)
    min_out   = np.array(min_out)
    max_out   = np.array(max_out)

    print(f"\nδ media final : {mean_out[-1]:.4f}")
    print(f"δ mínimo final: {min_out[-1]:.4f}")
    print(f"δ máximo final: {max_out[-1]:.4f}")

    # ── Step en el que δ_media cruza el umbral (interpolación lineal) ─
    above = np.where(mean_out >= threshold)[0]
    if len(above) == 0:
        print(f"\nδ={threshold:.2f}  ->  no alcanzado (δ_media máx = {mean_out.max():.4f})")
    else:
        k = above[0]
        if k == 0:
            step_cross = steps_out[0]
        else:
            x0, x1 = steps_out[k-1], steps_out[k]
            y0, y1 = mean_out[k-1], mean_out[k]
            step_cross = x0 + (threshold - y0) * (x1 - x0) / (y1 - y0)
        print(f"\nδ={threshold:.2f}  ->  step ≈ {step_cross:.1f}")

    # ── Plot temporal ──────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps_out, mean_out, color="tab:red", lw=1.8, label="δ media")
    ax.axhline(threshold, color="gray", ls=":", lw=1, alpha=0.6, label=f"δ={threshold:.2f}")
    ax.set_xlabel("Steps")
    ax.set_ylabel("δ (Lindemann, por pares)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Mapa espacial opcional ───────────────────────────────────
    if plot_spatial:
        delta_pair_final = np.sqrt(np.maximum(M2_r / count, 0)) / mean_r
        lindemann_atom = np.zeros(len(ref_pos))
        counts_atom = np.zeros(len(ref_pos))
        np.add.at(lindemann_atom, i_idx, delta_pair_final)
        np.add.at(lindemann_atom, j_idx, delta_pair_final)
        np.add.at(counts_atom, i_idx, 1)
        np.add.at(counts_atom, j_idx, 1)
        lindemann_atom = np.divide(lindemann_atom, counts_atom,
                                    out=np.zeros_like(lindemann_atom), where=counts_atom > 0)

        fig, ax = plt.subplots(figsize=(7, 6))
        sc = ax.scatter(ref_pos[:, 0], ref_pos[:, 1], c=lindemann_atom,
                         cmap="inferno", s=40, edgecolors="k", linewidths=0.3)
        ax.set_xlabel("x (Å)")
        ax.set_ylabel("y (Å)")
        ax.set_title("Distribución espacial de δ por átomo (acumulado)")
        ax.set_aspect("equal")
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label("δ por átomo")
        plt.tight_layout()
        plt.show()

    return steps_out, mean_out, min_out, max_out   # (única línea cambiada: devuelve los datos para el plot conjunto)


In [ ]:
# Celda 3 · Lindemann estático

def lindemann_estatico(fname, save_interval, eval_every, rcut=3.2,
                       in_plane=True, plot=True, plot_spatial=False):
    """
    Calcula el parámetro de Lindemann estático 2D (Bedanov et al., 1985) a lo largo de un .traj.

        γ_m,ij = < |u_i − u_j|² > / r0_ij²

    u es el desplazamiento respecto a la posición media, así que γ_m,ij es la varianza
    temporal del vector relativo i→j (desenvuelto), normalizada con la distancia del par
    en el frame 0. Promedio acumulado desde el frame 0, igual que el δ por pares.

    Parámetros
    ----------
    fname : str
        Nombre base del archivo (sin extensión .traj)
    save_interval : int
        Pasos de MD entre frames guardados
    eval_every : int
        Cada cuántos frames se evalúa/guarda γ_m
    rcut : float
        Radio de corte para definir pares vecinos (Å)
    in_plane : bool
        Si True usa solo las componentes x, y del desplazamiento (definición 2D)
    plot : bool
        Si True, plotea la evolución temporal
    plot_spatial : bool
        Si True, además plotea el mapa espacial de γ_m por átomo (acumulado, corrida completa)
    """
    traj = Trajectory(fname + ".traj")
    n_frames = len(traj)
    print(f"Frames totales: {n_frames}")

    ref_atoms = traj[0]
    cell, pbc_mask = ref_atoms.get_cell(), np.array(ref_atoms.get_pbc(), float)
    ref_pos = ref_atoms.get_positions()
    comps = [0, 1] if in_plane else [0, 1, 2]

    i_idx, j_idx = neighbor_list("ij", ref_atoms, cutoff=rcut)
    keep = i_idx < j_idx
    i_idx, j_idx = i_idx[keep], j_idx[keep]
    print(f"Pares vecinos (rcut={rcut} Å): {len(i_idx)}")

    def mic(d):
        frac = np.linalg.solve(cell.T, d.T).T
        frac -= np.round(frac) * pbc_mask
        return frac @ cell

    w = mic(ref_pos[j_idx] - ref_pos[i_idx])     # vector relativo i→j en el frame 0
    r0 = np.linalg.norm(w, axis=1)               # distancia de cada par en el frame 0
    prev = ref_pos.copy()

    # ── Welford online por par (sobre el vector relativo) ────────
    mean_w = np.zeros((len(i_idx), len(comps)))
    M2_w = np.zeros((len(i_idx), len(comps)))
    steps_out, mean_out, min_out, max_out = [], [], [], []

    for count, f in enumerate(range(n_frames), start=1):
        pos = traj[f].positions
        salto = mic(pos - prev)                  # desplazamiento de cada átomo desde el frame anterior
        w = w + salto[j_idx] - salto[i_idx]      # vector relativo desenvuelto (sin plegado)
        prev = pos.copy()

        x = w[:, comps]
        d = x - mean_w
        mean_w = mean_w + d / count
        M2_w = M2_w + d * (x - mean_w)

        if count % eval_every == 0 or f == n_frames - 1:
            gamma_pair = np.sum(M2_w / count, axis=1) / r0**2
            steps_out.append(f * save_interval)
            mean_out.append(gamma_pair.mean())
            min_out.append(gamma_pair.min())
            max_out.append(gamma_pair.max())

    steps_out = np.array(steps_out)
    mean_out  = np.array(mean_out)
    min_out   = np.array(min_out)
    max_out   = np.array(max_out)

    print(f"\nγ_m media final : {mean_out[-1]:.4f}")
    print(f"γ_m mínimo final: {min_out[-1]:.4f}")
    print(f"γ_m máximo final: {max_out[-1]:.4f}")

    # ── Plot temporal ────────────────────────────────────────────
    if plot:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(steps_out, mean_out, color="#2a78d6", lw=1.8, label="γ_m media")
        ax.set_xlabel("Steps")
        ax.set_ylabel("γ_m (Lindemann estático)")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Mapa espacial opcional ───────────────────────────────────
    if plot_spatial:
        lindemann_atom = np.zeros(len(ref_pos))
        counts_atom = np.zeros(len(ref_pos))
        np.add.at(lindemann_atom, i_idx, gamma_pair)
        np.add.at(lindemann_atom, j_idx, gamma_pair)
        np.add.at(counts_atom, i_idx, 1)
        np.add.at(counts_atom, j_idx, 1)
        lindemann_atom = np.divide(lindemann_atom, counts_atom,
                                   out=np.zeros_like(lindemann_atom), where=counts_atom > 0)

        fig, ax = plt.subplots(figsize=(7, 6))
        sc = ax.scatter(ref_pos[:, 0], ref_pos[:, 1], c=lindemann_atom,
                        cmap="inferno", s=40, edgecolors="k", linewidths=0.3)
        ax.set_xlabel("x (Å)")
        ax.set_ylabel("y (Å)")
        ax.set_title("Distribución espacial de γ_m por átomo (acumulado)")
        ax.set_aspect("equal")
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label("γ_m por átomo")
        plt.tight_layout()
        plt.show()

    return steps_out, mean_out, min_out, max_out


In [ ]:
# Celda 4 · Lindemann dinámico

def lindemann_dinamico(fname, save_interval, eval_every, rcut=3.2,
                       in_plane=True, plot=True, plot_spatial=False):
    """
    Calcula el parámetro de Lindemann dinámico 2D (Zahn y Maret, 2000) a lo largo de un .traj.

        γ_L,ij(t) = < |Δu_i(t) − Δu_j(t)|² > / (2 r0_ij²),     Δu(t) = R(t) − R(0)

    Es el cambio del vector relativo i→j (desenvuelto) desde el frame 0, normalizado con
    la distancia del par en el frame 0. En el cristal llega a una meseta; si el sistema
    funde, crece sin límite.

    Parámetros
    ----------
    fname : str
        Nombre base del archivo (sin extensión .traj)
    save_interval : int
        Pasos de MD entre frames guardados
    eval_every : int
        Cada cuántos frames se evalúa/guarda γ_L
    rcut : float
        Radio de corte para definir pares vecinos (Å)
    in_plane : bool
        Si True usa solo las componentes x, y del desplazamiento (definición 2D)
    plot : bool
        Si True, plotea la evolución temporal
    plot_spatial : bool
        Si True, además plotea el mapa espacial de γ_L por átomo (último frame)
    """
    traj = Trajectory(fname + ".traj")
    n_frames = len(traj)
    print(f"Frames totales: {n_frames}")

    ref_atoms = traj[0]
    cell, pbc_mask = ref_atoms.get_cell(), np.array(ref_atoms.get_pbc(), float)
    ref_pos = ref_atoms.get_positions()
    comps = [0, 1] if in_plane else [0, 1, 2]

    i_idx, j_idx = neighbor_list("ij", ref_atoms, cutoff=rcut)
    keep = i_idx < j_idx
    i_idx, j_idx = i_idx[keep], j_idx[keep]
    print(f"Pares vecinos (rcut={rcut} Å): {len(i_idx)}")

    def mic(d):
        frac = np.linalg.solve(cell.T, d.T).T
        frac -= np.round(frac) * pbc_mask
        return frac @ cell

    w = mic(ref_pos[j_idx] - ref_pos[i_idx])     # vector relativo i→j en el frame 0
    r0 = np.linalg.norm(w, axis=1)               # distancia de cada par en el frame 0
    w0 = w[:, comps].copy()                      # origen t = 0
    prev = ref_pos.copy()

    steps_out, mean_out, min_out, max_out = [], [], [], []

    for count, f in enumerate(range(n_frames), start=1):
        pos = traj[f].positions
        salto = mic(pos - prev)                  # desplazamiento de cada átomo desde el frame anterior
        w = w + salto[j_idx] - salto[i_idx]      # vector relativo desenvuelto (sin plegado)
        prev = pos.copy()

        if count % eval_every == 0 or f == n_frames - 1:
            gamma_pair = np.sum((w[:, comps] - w0)**2, axis=1) / (2 * r0**2)
            steps_out.append(f * save_interval)
            mean_out.append(gamma_pair.mean())
            min_out.append(gamma_pair.min())
            max_out.append(gamma_pair.max())

    steps_out = np.array(steps_out)
    mean_out  = np.array(mean_out)
    min_out   = np.array(min_out)
    max_out   = np.array(max_out)

    print(f"\nγ_L media final : {mean_out[-1]:.4f}")
    print(f"γ_L mínimo final: {min_out[-1]:.4f}")
    print(f"γ_L máximo final: {max_out[-1]:.4f}")

    # ── Plot temporal ────────────────────────────────────────────
    if plot:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(steps_out, mean_out, color="#1baf7a", lw=1.8, label="γ_L media")
        ax.set_xlabel("Steps")
        ax.set_ylabel("γ_L (Lindemann dinámico)")
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Mapa espacial opcional ───────────────────────────────────
    if plot_spatial:
        lindemann_atom = np.zeros(len(ref_pos))
        counts_atom = np.zeros(len(ref_pos))
        np.add.at(lindemann_atom, i_idx, gamma_pair)
        np.add.at(lindemann_atom, j_idx, gamma_pair)
        np.add.at(counts_atom, i_idx, 1)
        np.add.at(counts_atom, j_idx, 1)
        lindemann_atom = np.divide(lindemann_atom, counts_atom,
                                   out=np.zeros_like(lindemann_atom), where=counts_atom > 0)

        fig, ax = plt.subplots(figsize=(7, 6))
        sc = ax.scatter(ref_pos[:, 0], ref_pos[:, 1], c=lindemann_atom,
                        cmap="inferno", s=40, edgecolors="k", linewidths=0.3)
        ax.set_xlabel("x (Å)")
        ax.set_ylabel("y (Å)")
        ax.set_title("Distribución espacial de γ_L por átomo (último frame)")
        ax.set_aspect("equal")
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label("γ_L por átomo")
        plt.tight_layout()
        plt.show()

    return steps_out, mean_out, min_out, max_out


# Utilidades de vecinos

def vecinos_promedio(fname, rcut):
    """Número medio de vecinos por átomo con el radio de corte rcut (frame 0)."""
    atoms = Trajectory(fname + ".traj")[0]
    i_idx = neighbor_list("i", atoms, cutoff=rcut)
    n_vec = np.bincount(i_idx, minlength=len(atoms))
    print(f"rcut = {rcut:.3f} Å -> vecinos promedio = {n_vec.mean():.2f} "
          f"(mín {n_vec.min()}, máx {n_vec.max()})")
    return n_vec.mean()


def rcut_para_vecinos(fname, n_vecinos, rmax=8.0):
    """
    Radio de corte que da 'n_vecinos' vecinos por átomo (frame 0): punto medio entre
    la distancia media al vecino n y al vecino n+1 de cada átomo.
    """
    atoms = Trajectory(fname + ".traj")[0]
    i_idx, dist = neighbor_list("id", atoms, cutoff=rmax)
    d_n, d_n1 = [], []
    for k in range(len(atoms)):
        dk = np.sort(dist[i_idx == k])
        if len(dk) > n_vecinos:
            d_n.append(dk[n_vecinos - 1])
            d_n1.append(dk[n_vecinos])
    d_n, d_n1 = np.mean(d_n), np.mean(d_n1)
    rc = 0.5 * (d_n + d_n1)
    print(f"{n_vecinos} vecinos: vecino {n_vecinos} a {d_n:.3f} Å, vecino {n_vecinos + 1} a {d_n1:.3f} Å "
          f"-> rcut = {rc:.3f} Å")
    return rc


In [ ]:
# Celda 5 · g(r) total y parciales del frame inicial

def rdf_inicial(fname, frame=0, rmax=6.0, dr=0.01, sigma=0.03, rcut_previo=None,
                n_capas=5, plot=True):
    """
    g(r) total y parciales (por pares de elementos) de un frame del .traj, y número de
    vecinos acumulado n(r). Propone radios de corte en los mínimos de la g(r) total.

    Normalización 2D (lámina con PBC en x, y y vacío en z):
        g(r) = n_pares(r) · A / (N_a · N_b · 2π r dr),   A = área de la celda en el plano xy
    El número de vecinos acumulado no depende de la normalización: n(r) es el número
    medio de vecinos a distancia < r, es decir, lo que daría rcut = r.

    Parámetros
    ----------
    fname : str
        Nombre base del archivo (sin extensión .traj)
    frame : int
        Frame que se analiza (0 = posición inicial)
    rmax : float
        Distancia máxima de la g(r) (Å)
    dr : float
        Anchura de los intervalos del histograma (Å)
    sigma : float
        Anchura del suavizado gaussiano (Å). Ayuda a localizar mínimos si el frame
        está desordenado térmicamente; 0 = sin suavizar
    rcut_previo : float o None
        Si se da, se marca en las gráficas para comparar (p. ej. tu 3.2 Å)
    n_capas : int
        Número máximo de radios de corte propuestos
    plot : bool
        Si True, dibuja la g(r) total, las parciales y n(r)
    """
    atoms = Trajectory(fname + ".traj")[frame]
    N = len(atoms)
    cell = atoms.get_cell()
    area = np.linalg.norm(np.cross(cell[0], cell[1]))
    sym = np.array(atoms.get_chemical_symbols())
    especies = sorted(set(sym.tolist()))
    print(f"Frame {frame}: {N} átomos, especies {especies}, área xy = {area:.2f} Å², pbc = {atoms.get_pbc()}")

    i_idx, j_idx, d = neighbor_list("ijd", atoms, cutoff=rmax)
    edges = np.arange(0.0, rmax + dr, dr)
    r = 0.5 * (edges[1:] + edges[:-1])
    anillo = 2 * np.pi * r * dr

    def suavizar(y):
        if sigma <= 0:
            return y
        x = np.arange(-4 * sigma, 4 * sigma + dr, dr)
        w = np.exp(-0.5 * (x / sigma) ** 2)
        return np.convolve(y, w / w.sum(), mode="same")

    # ── g(r) y n(r) totales ──────────────────────────────────────
    hist, _ = np.histogram(d, bins=edges)
    g_tot = suavizar(hist * area / (N * N * anillo))
    n_tot = np.cumsum(hist) / N

    # ── Vecinos acumulados por especie central (todos los vecinos) ─
    n_esp = {}
    for a in especies:
        h, _ = np.histogram(d[sym[i_idx] == a], bins=edges)
        n_esp[a] = np.cumsum(h) / (sym == a).sum()

    # ── Parciales g_ab(r) y n_ab(r) (vecinos b alrededor de a) ───
    parciales = {}
    for k, a in enumerate(especies):
        for b in especies[k:]:
            m = (sym[i_idx] == a) & (sym[j_idx] == b)
            h, _ = np.histogram(d[m], bins=edges)
            Na, Nb = (sym == a).sum(), (sym == b).sum()
            parciales[f"{a}–{b}"] = {"g": suavizar(h * area / (Na * Nb * anillo)),
                                     "n_ab": np.cumsum(h) / Na,     # b alrededor de a
                                     "n_ba": np.cumsum(h) / Nb,     # a alrededor de b
                                     "a": a, "b": b}

    # ── Mínimos de la g(r) total -> radios de corte candidatos ───
    gmax = g_tot.max()
    eps = 1e-3 * gmax
    k0 = int(np.argmax(g_tot > 0.1 * gmax))             # inicio de la primera capa
    minimos, k, n = [], k0 + 1, len(g_tot)
    while k < n - 1:
        if g_tot[k] <= eps:                              # hueco vacío entre capas: su centro
            k1 = k
            while k1 < n - 1 and g_tot[k1] <= eps:
                k1 += 1
            if k1 < n - 1:
                minimos.append((k + k1 - 1) // 2)
            k = k1
        elif g_tot[k] < g_tot[k - 1] and g_tot[k] <= g_tot[k + 1]:
            minimos.append(k)
            k += 1
        else:
            k += 1
    # quedarse con mínimos marcados: más bajos que la mitad de los picos que los rodean
    bordes = [k0] + minimos + [n - 1]
    rcuts = []
    for p, km in enumerate(minimos, start=1):
        izq = g_tot[bordes[p - 1]:km + 1].max()
        der = g_tot[km:bordes[p + 1] + 1].max()
        if der > eps and g_tot[km] < 0.5 * min(izq, der):
            rcuts.append(r[km])
    rcuts = np.array(rcuts[:n_capas])
    vecinos = np.array([n_tot[np.searchsorted(r, rc)] for rc in rcuts])

    print(f"\n{'capa':>5}{'rcut (Å)':>10}{'vecinos':>9}" + "".join(f"{'de ' + a:>9}" for a in especies))
    for c, (rc, nv) in enumerate(zip(rcuts, vecinos), start=1):
        ki = np.searchsorted(r, rc)
        print(f"{c:>5}{rc:>10.3f}{nv:>9.2f}" + "".join(f"{n_esp[a][ki]:>9.2f}" for a in especies))
    if rcut_previo is not None:
        ki = np.searchsorted(r, rcut_previo)
        print(f"{'previo':>5}{rcut_previo:>10.3f}{n_tot[ki]:>9.2f}"
              + "".join(f"{n_esp[a][ki]:>9.2f}" for a in especies))
    print("(vecinos = número medio de vecinos por átomo; 'de X' = vecinos medios de los átomos X)")

    # ── Plots ────────────────────────────────────────────────────
    if plot:
        def marcar(ax, texto=True):
            for c, (rc, nv) in enumerate(zip(rcuts, vecinos), start=1):
                ax.axvline(rc, color="0.45", ls=":", lw=1)
                if texto:
                    ax.text(rc, 0.97, f" {c}: {nv:.1f}", transform=ax.get_xaxis_transform(),
                            va="top", fontsize=8, color="0.3")
            if rcut_previo is not None:
                ax.axvline(rcut_previo, color="tab:red", ls="--", lw=1.2, label=f"rcut previo = {rcut_previo} Å")

        fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
        axs[0].plot(r, g_tot, color="0.15", lw=1.5, label="g(r) total")
        marcar(axs[0])
        axs[0].set_ylabel("g(r)")
        axs[0].set_title("g(r) total del frame inicial · líneas punteadas = mínimos (capa: vecinos)", fontsize=11)
        axs[1].plot(r, n_tot, color="0.15", lw=1.5, label="todos los átomos")
        for a, color in zip(especies, ["#2a78d6", "#eb6834", "#1baf7a"]):
            if len(especies) > 1:
                axs[1].plot(r, n_esp[a], color=color, lw=1.3, ls="--", label=f"átomos {a}")
        marcar(axs[1], texto=False)
        axs[1].set_xlabel("r (Å)")
        axs[1].set_ylabel("vecinos con distancia < r")
        axs[1].set_title("Número de vecinos acumulado = vecinos que da rcut = r", fontsize=11)
        for ax in axs:
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        if len(especies) > 1:
            colores = ["#2a78d6", "#eb6834", "#1baf7a", "tab:purple", "tab:brown", "tab:gray"]
            fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
            for (par, p), color in zip(parciales.items(), colores):
                axs[0].plot(r, p["g"], color=color, lw=1.5, label=f"g {par}")
                axs[1].plot(r, p["n_ab"], color=color, lw=1.5, label=f"{p['b']} alrededor de {p['a']}")
                if p["a"] != p["b"]:
                    axs[1].plot(r, p["n_ba"], color=color, lw=1.5, ls="--", label=f"{p['a']} alrededor de {p['b']}")
            marcar(axs[0])
            marcar(axs[1], texto=False)
            axs[0].set_ylabel("g_ab(r)")
            axs[0].set_title("g(r) parciales por pares de elementos", fontsize=11)
            axs[1].set_xlabel("r (Å)")
            axs[1].set_ylabel("vecinos con distancia < r")
            axs[1].set_title("Vecinos acumulados por tipo de par", fontsize=11)
            for ax in axs:
                ax.grid(alpha=0.3)
                ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
        else:
            print("Una sola especie: la g(r) parcial coincide con la total.")

    return {"r": r, "g": g_tot, "n": n_tot, "n_especie": n_esp, "parciales": parciales,
            "rcut_sugeridos": rcuts, "vecinos": vecinos}


## Parámetros

In [ ]:
# Celda 6 · parámetros
fname         = "mi_trayectoria"   # nombre del .traj SIN la extensión
save_interval = 10                 # pasos de MD entre frames guardados
eval_every    = 50                 # cada cuántos frames se evalúa
in_plane      = True               # estático y dinámico solo con x, y

rmax_gr       = 6.0                # distancia máxima de la g(r) (Å)
rcut_previo   = 3.2                # tu radio de corte anterior, para compararlo en la g(r)

## Paso 1 · g(r) del frame inicial y elección del radio de corte

La tabla da, para cada mínimo de la g(r) total, el `rcut` propuesto y cuántos vecinos medios incluye (en total y por especie). Las gráficas muestran:
- la g(r) total y el número de vecinos acumulado: el valor de la curva en r es lo que daría `rcut = r`; en una red limpia aparecen mesetas entre capas;
- si hay varias especies (p. ej. borografdiino), las g(r) parciales C–C, B–C, B–B y los vecinos acumulados de cada tipo.

Un buen `rcut` cae en un mínimo de la g(r), donde el número de vecinos apenas cambia: así pequeñas vibraciones en el frame 0 no meten ni sacan pares de la lista.

In [ ]:
# Celda 7 · g(r) del frame inicial
gr = rdf_inicial(fname, frame=0, rmax=rmax_gr, rcut_previo=rcut_previo)

In [ ]:
# Celda 8 · elegir el radio de corte
capa = 1        # 1 = primer mínimo (solo vecinos enlazados), 2 = incluye segundos vecinos (sensible a ángulos), ...
rcut = round(float(gr["rcut_sugeridos"][capa - 1]), 3)
# rcut = 3.2    # o pon tu valor a mano

print(f"rcut elegido: {rcut} Å")
n_vec_rcut = vecinos_promedio(fname, rcut)

## Opcional · δ por pares para cada capa de la g(r)

Ejecuta tu función con el `rcut` de cada mínimo y superpone las curvas, para ver cómo cambia δ al incluir más capas de vecinos. Los plots individuales se omiten en esta celda.

In [ ]:
# Celda 9 · δ por pares según la capa de vecinos
n_capas_comparar = 3

_show = plt.show
plt.show = lambda *a, **k: plt.close("all")        # silencia los plots individuales de tu función
res_capas = {}
try:
    for k, rc in enumerate(gr["rcut_sugeridos"][:n_capas_comparar], start=1):
        print(f"\n── capa {k}: rcut = {rc:.3f} Å ({gr['vecinos'][k - 1]:.1f} vecinos) ──")
        res_capas[k] = lindemann_analysis(fname, save_interval, eval_every, rcut=round(float(rc), 3))
finally:
    plt.show = _show

colores = plt.get_cmap("Reds")(np.linspace(0.45, 0.95, len(res_capas)))
fig, ax = plt.subplots(figsize=(8, 5))
for (k, (steps, media, mn, mx)), color in zip(res_capas.items(), colores):
    ax.plot(steps, media, color=color, lw=1.8,
            label=f"capa {k}: rcut = {gr['rcut_sugeridos'][k - 1]:.2f} Å ({gr['vecinos'][k - 1]:.1f} vecinos)")
ax.set_xlabel("Steps")
ax.set_ylabel("δ (Lindemann, por pares)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Opcional · comparar tres radios de corte dados

Pon tus radios en `rcuts_comparar`. Para cada uno calcula el número medio de vecinos y los tres métodos (por pares con tu función, estático y dinámico), y los superpone en una figura de tres paneles, con una tabla de valores finales. Mejor si cada radio cae en un mínimo de la g(r) de la celda 7.

In [ ]:
# Celda 9b · comparar tres radios de corte dados
rcuts_comparar = [2.0, 2.7, 3.2]        # pon aquí tus radios de corte (Å)

_show = plt.show
plt.show = lambda *a, **k: plt.close("all")        # silencia los plots individuales de cada función
res_p, res_e, res_d, n_vec = {}, {}, {}, {}
try:
    for rc in rcuts_comparar:
        print(f"\n── rcut = {rc} Å ──")
        n_vec[rc] = vecinos_promedio(fname, rc)
        res_p[rc] = lindemann_analysis(fname, save_interval, eval_every, rcut=rc)
        res_e[rc] = lindemann_estatico(fname, save_interval, eval_every, rcut=rc, in_plane=in_plane, plot=False)
        res_d[rc] = lindemann_dinamico(fname, save_interval, eval_every, rcut=rc, in_plane=in_plane, plot=False)
finally:
    plt.show = _show

colores = plt.get_cmap("Blues")(np.linspace(0.45, 0.95, len(rcuts_comparar)))
estilos = ["-", "--", "-.", ":"]
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, res, titulo in ((axs[0], res_p, "δ por pares"), (axs[1], res_e, "γ_m estático"), (axs[2], res_d, "γ_L dinámico")):
    for k, rc in enumerate(rcuts_comparar):
        steps, media = res[rc][0], res[rc][1]
        ax.plot(steps, media, color=colores[k], ls=estilos[k % len(estilos)], lw=1.8,
                label=f"rcut = {rc} Å ({n_vec[rc]:.1f} vecinos)")
    ax.set_xlabel("Steps")
    ax.set_ylabel(titulo)
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'rcut (Å)':>9}{'vecinos':>9}{'δ final':>10}{'γ_m final':>11}{'γ_L final':>11}")
for rc in rcuts_comparar:
    print(f"{rc:>9}{n_vec[rc]:>9.2f}{res_p[rc][1][-1]:>10.4f}{res_e[rc][1][-1]:>11.4f}{res_d[rc][1][-1]:>11.4f}")

## 1 · Lindemann por pares

In [ ]:
# Celda 10 · por pares (tu función)
steps_p, mean_p, min_p, max_p = lindemann_analysis(fname, save_interval, eval_every, rcut=rcut)

## 2 · Lindemann estático

In [ ]:
# Celda 11 · estático
steps_e, mean_e, min_e, max_e = lindemann_estatico(fname, save_interval, eval_every, rcut=rcut,
                                                   in_plane=in_plane)

## 3 · Lindemann dinámico

In [ ]:
# Celda 12 · dinámico
steps_d, mean_d, min_d, max_d = lindemann_dinamico(fname, save_interval, eval_every, rcut=rcut,
                                                   in_plane=in_plane)

## 4 · Plot conjunto

Escala logarítmica en y porque las tres magnitudes tienen órdenes distintos (en el sólido, γ_m ≈ γ_L ≈ 2δ²).

In [ ]:
# Celda 13 · plot conjunto
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(steps_p, mean_p, color="tab:red", lw=1.8, label="δ por pares")
ax.plot(steps_e, mean_e, color="#2a78d6", lw=1.8, ls="--", label="γ_m estático")
ax.plot(steps_d, mean_d, color="#1baf7a", lw=1.8, ls="-.", label="γ_L dinámico")
ax.set_yscale("log")
ax.set_xlabel("Steps")
ax.set_ylabel("Parámetro de Lindemann")
ax.set_title(f"rcut = {rcut} Å ({n_vec_rcut:.1f} vecinos de media)")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## 5 · Estático y dinámico con 9 y 12 vecinos

Se busca el radio de corte que da 9 y 12 vecinos por átomo y se compara la evolución con la de tu `rcut`. Cada par se normaliza con su propia distancia r0, así que los resultados con distinto número de vecinos son comparables.

In [ ]:
# Celda 14 · comparación por número de vecinos
n_comparar = [9, 12]

casos = [(f"rcut = {rcut} Å ({n_vec_rcut:.1f} vecinos)", rcut)]
for n in n_comparar:
    rc = round(rcut_para_vecinos(fname, n), 3)
    n_real = vecinos_promedio(fname, rc)
    casos.append((f"{n} vecinos (rcut = {rc:.2f} Å, media {n_real:.1f})", rc))

res_e, res_d = {}, {}
for etiqueta, rc in casos:
    print(f"\n── {etiqueta} ──")
    res_e[etiqueta] = lindemann_estatico(fname, save_interval, eval_every, rcut=rc, in_plane=in_plane, plot=False)
    res_d[etiqueta] = lindemann_dinamico(fname, save_interval, eval_every, rcut=rc, in_plane=in_plane, plot=False)

colores = plt.get_cmap("Blues")(np.linspace(0.45, 0.95, len(casos)))
estilos = ["-", "--", "-."]
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
for ax, res, titulo in ((axs[0], res_e, "γ_m estático"), (axs[1], res_d, "γ_L dinámico")):
    for (etiqueta, _), color, ls in zip(casos, colores, estilos):
        steps, media = res[etiqueta][0], res[etiqueta][1]
        ax.plot(steps, media, color=color, ls=ls, lw=1.8, label=etiqueta)
    ax.set_yscale("log")
    ax.set_xlabel("Steps")
    ax.set_ylabel(titulo)
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"\n{'Caso':<45}{'γ_m final':>11}{'γ_L final':>11}")
for etiqueta, _ in casos:
    print(f"{etiqueta:<45}{res_e[etiqueta][1][-1]:>11.4f}{res_d[etiqueta][1][-1]:>11.4f}")

## 6 · Lindemann local: media local frente a media del sistema (Zhang et al., 2007)

Mismo δ_ij que tu función por pares, agrupado por átomo:

| Magnitud | Definición |
|---|---|
| **q_i** (índice local del átomo i) | media de los δ_ij de los pares en los que participa i |
| **q del sistema** | media de q_i sobre **todos** los átomos |
| **q local** | media de q_i solo sobre una **selección** de átomos (en Zhang et al.: los átomos de un defecto Stone–Wales y sus primeros vecinos) |

La media del sistema coincide con tu δ (media sobre pares) cuando todos los átomos tienen el mismo número de pares: con todos los pares, como en Zhang et al., o con una coordinación uniforme. La media local sirve para detectar que **una región** empieza a fundir (premelting) aunque la media del sistema apenas cambie, porque esos átomos son pocos y su señal queda diluida.

En Zhang et al. j recorre todos los átomos; aquí, por defecto, los vecinos de tu `rcut` (`todos_los_pares=True` reproduce la definición original, solo para sistemas pequeños).

In [ ]:
# Celda 15 · Lindemann local y utilidades de selección

def lindemann_local(fname, save_interval, eval_every, rcut=3.2, seleccion=None,
                    todos_los_pares=False, q_c=None, plot=True):
    """
    Índice de Lindemann local por átomo y sus promedios (Zhang et al., Nanotechnology 18,
    285703, 2007), a lo largo de un .traj:

        q_i        = (1/n_i) · Σ_j δ_ij      media de los δ_ij de los pares en los que está i
        q_sistema  = (1/N)   · Σ_i q_i       media sobre todos los átomos
        q_local    = media de q_i sobre los átomos seleccionados (p. ej. un defecto + vecinos)

    δ_ij es el mismo que en tu función por pares: sqrt(<r²> − <r>²) / <r>, con imagen
    mínima en cada frame y promedio acumulado desde el frame 0.
    En Zhang et al. j recorre todos los demás átomos (n_i = N − 1). Aquí, por defecto, los
    vecinos con r < rcut en el frame 0; todos_los_pares=True reproduce la definición original
    (coste ∝ N², pensado para sistemas pequeños).

    Parámetros
    ----------
    fname : str
        Nombre base del archivo (sin extensión .traj)
    save_interval : int
        Pasos de MD entre frames guardados
    eval_every : int
        Cada cuántos frames se evalúa/guarda
    rcut : float
        Radio de corte para definir pares vecinos (Å). Se ignora con todos_los_pares=True
    seleccion : array de índices, máscara booleana o None
        Átomos que forman la región "local". Ver seleccion_por_region/especie/coordinacion
    todos_los_pares : bool
        Si True usa todos los pares i<j (definición original de Zhang et al.)
    q_c : float o None
        Si se da, calcula además la fracción de "átomos de Lindemann" (q_i > q_c)
    plot : bool
        Si True, dibuja la comparación y el mapa espacial de q_i
    """
    traj = Trajectory(fname + ".traj")
    n_frames = len(traj)
    print(f"Frames totales: {n_frames}")

    ref_atoms = traj[0]
    N = len(ref_atoms)
    cell, pbc_mask = ref_atoms.get_cell(), np.array(ref_atoms.get_pbc(), float)
    ref_pos = ref_atoms.get_positions()

    if todos_los_pares:
        i_idx, j_idx = np.triu_indices(N, k=1)
        print(f"Todos los pares: {len(i_idx)} (N = {N})")
        if len(i_idx) > 5e6:
            print("AVISO: muchos pares; el cálculo será lento y usará mucha memoria.")
    else:
        i_idx, j_idx = neighbor_list("ij", ref_atoms, cutoff=rcut)
        keep = i_idx < j_idx
        i_idx, j_idx = i_idx[keep], j_idx[keep]
        print(f"Pares vecinos (rcut={rcut} Å): {len(i_idx)}")

    n_pares_atomo = np.bincount(i_idx, minlength=N) + np.bincount(j_idx, minlength=N)
    validos = n_pares_atomo > 0

    sel = np.zeros(N, bool)
    if seleccion is not None:
        seleccion = np.asarray(seleccion)
        if seleccion.dtype == bool:
            sel[:] = seleccion
        else:
            sel[seleccion.astype(int)] = True
    print(f"Átomos en la selección local: {sel.sum()} de {N}")

    def r_ij(atoms):
        d = atoms.positions[j_idx] - atoms.positions[i_idx]
        frac = np.linalg.solve(cell.T, d.T).T
        frac -= np.round(frac) * pbc_mask
        return np.linalg.norm(frac @ cell, axis=1)

    def q_por_atomo(delta_pair):
        q = np.zeros(N)
        np.add.at(q, i_idx, delta_pair)
        np.add.at(q, j_idx, delta_pair)
        return np.divide(q, n_pares_atomo, out=np.zeros(N), where=validos)

    # ── Welford online por par (igual que tu función) ─────────────
    mean_r, M2_r = np.zeros(len(i_idx)), np.zeros(len(i_idx))
    steps_out, pares_out, sist_out, loc_out, resto_out, frac_out = [], [], [], [], [], []

    for count, f in enumerate(range(n_frames), start=1):
        r = r_ij(traj[f])
        d = r - mean_r
        mean_r = mean_r + d / count
        M2_r = M2_r + d * (r - mean_r)

        if count % eval_every == 0 or f == n_frames - 1:
            delta_pair = np.sqrt(np.maximum(M2_r / count, 0)) / mean_r
            q = q_por_atomo(delta_pair)
            steps_out.append(f * save_interval)
            pares_out.append(delta_pair.mean())                       # tu δ: media sobre pares
            sist_out.append(q[validos].mean())                        # Zhang: media sobre átomos
            loc_out.append(q[sel & validos].mean() if (sel & validos).any() else np.nan)
            resto_out.append(q[~sel & validos].mean() if (~sel & validos).any() else np.nan)
            if q_c is not None:
                frac_out.append((q[validos] > q_c).mean())

    steps_out = np.array(steps_out)
    pares_out, sist_out = np.array(pares_out), np.array(sist_out)
    loc_out, resto_out = np.array(loc_out), np.array(resto_out)

    print(f"\nδ media sobre pares (tu función) : {pares_out[-1]:.4f}")
    print(f"q media del sistema (átomos)     : {sist_out[-1]:.4f}")
    if sel.any():
        print(f"q media local (selección)        : {loc_out[-1]:.4f}")
        print(f"q media resto del sistema        : {resto_out[-1]:.4f}")
    if q_c is not None:
        print(f"Fracción de átomos con q_i > {q_c}: {frac_out[-1]:.3f}")

    # ── Plots ────────────────────────────────────────────────────
    if plot:
        fig, axs = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.3, 1]})
        ax = axs[0]
        ax.plot(steps_out, sist_out, color="#2a78d6", lw=2, label="q media del sistema (todos los átomos)")
        if sel.any():
            ax.plot(steps_out, loc_out, color="#eb6834", lw=2, label=f"q media local ({sel.sum()} átomos)")
            ax.plot(steps_out, resto_out, color="0.5", lw=1.5, ls="--", label="q media del resto")
        ax.plot(steps_out, pares_out, color="0.1", lw=1.6, ls=":", zorder=5, label="δ media sobre pares (tu función)")
        ax.set_xlabel("Steps")
        ax.set_ylabel("Índice de Lindemann")
        ax.set_title("Media local frente a media del sistema", loc="left", fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

        ax = axs[1]
        q_final = q_por_atomo(delta_pair)
        sc = ax.scatter(ref_pos[:, 0], ref_pos[:, 1], c=q_final, cmap="inferno", s=40,
                        edgecolors="k", linewidths=0.3)
        if sel.any():
            ax.scatter(ref_pos[sel, 0], ref_pos[sel, 1], s=110, facecolors="none",
                       edgecolors="#1baf7a", linewidths=1.6, label="selección local")
            ax.legend(fontsize=9, loc="upper right")
        ax.set_xlabel("x (Å)")
        ax.set_ylabel("y (Å)")
        ax.set_title("q_i por átomo (final) y selección local", loc="left", fontsize=11)
        ax.set_aspect("equal")
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label("q_i")
        plt.tight_layout()
        plt.show()

        if q_c is not None:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(steps_out, frac_out, color="#eb6834", lw=1.8)
            ax.set_xlabel("Steps")
            ax.set_ylabel(f"fracción con q_i > {q_c}")
            ax.set_title("Fracción de átomos de Lindemann", loc="left", fontsize=11)
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

    return {"steps": steps_out, "delta_pares": pares_out, "q_sistema": sist_out,
            "q_local": loc_out, "q_resto": resto_out,
            "fraccion": np.array(frac_out) if q_c is not None else None,
            "q_i_final": q_por_atomo(delta_pair), "seleccion": sel}


# ── Utilidades para elegir la región local (sobre el frame 0) ────────

def seleccion_por_region(fname, centro_xy, radio):
    """Átomos a menos de 'radio' (Å) del punto centro_xy en el plano xy (con imagen mínima)."""
    atoms = Trajectory(fname + ".traj")[0]
    cell, pbc_mask = atoms.get_cell(), np.array(atoms.get_pbc(), float)
    d = atoms.positions - np.array([centro_xy[0], centro_xy[1], 0.0])
    d[:, 2] = 0.0
    frac = np.linalg.solve(cell.T, d.T).T
    frac -= np.round(frac) * pbc_mask
    dist = np.linalg.norm((frac @ cell)[:, :2], axis=1)
    return np.where(dist < radio)[0]


def seleccion_por_especie(fname, simbolos):
    """Átomos de la especie o especies indicadas, p. ej. "B" o ["B", "N"]."""
    atoms = Trajectory(fname + ".traj")[0]
    sym = np.array(atoms.get_chemical_symbols())
    return np.where(np.isin(sym, np.atleast_1d(simbolos)))[0]


def seleccion_por_coordinacion(fname, rcut_enlace, z):
    """Átomos con z vecinos enlazados (r < rcut_enlace), p. ej. z=2 para carbonos sp."""
    atoms = Trajectory(fname + ".traj")[0]
    n_vec = np.bincount(neighbor_list("i", atoms, cutoff=rcut_enlace), minlength=len(atoms))
    return np.where(np.isin(n_vec, np.atleast_1d(z)))[0]


def expandir_vecinos(fname, seleccion, rcut_enlace, capas=1):
    """Añade a la selección sus vecinos enlazados ('capas' veces), como el defecto + vecinos de Zhang et al."""
    atoms = Trajectory(fname + ".traj")[0]
    i_idx, j_idx = neighbor_list("ij", atoms, cutoff=rcut_enlace)
    sel = np.zeros(len(atoms), bool)
    sel[np.asarray(seleccion, int)] = True
    for _ in range(capas):
        sel[j_idx[sel[i_idx]]] = True
    return np.where(sel)[0]


In [ ]:
# Celda 16 · media local frente a media del sistema
# Elige qué átomos forman la región local (deja una opción sin comentar):
_c = Trajectory(fname + ".traj")[0].get_cell()
centro = 0.5 * (_c[0] + _c[1])[:2]                       # centro de la celda en el plano xy

seleccion = seleccion_por_region(fname, centro_xy=centro, radio=6.0)               # zona alrededor de un punto (p. ej. un defecto)
# seleccion = seleccion_por_especie(fname, "B")                                     # p. ej. los B del borografdiino
# seleccion = seleccion_por_coordinacion(fname, rcut_enlace=1.8, z=2)                # p. ej. carbonos sp de las cadenas del grafdiino
# seleccion = expandir_vecinos(fname, seleccion_por_coordinacion(fname, 1.8, [1, 2]), rcut_enlace=1.8)   # átomos poco coordinados + vecinos (como Zhang)

res_local = lindemann_local(fname, save_interval, eval_every, rcut=rcut, seleccion=seleccion)